In [ ]:
import math
from pathlib import Path
from urllib.parse import urljoin

import requests
import rasterio
from rasterio.windows import Window

import numpy as np
import matplotlib.pyplot as plt


def slice_visual_save_mask(
    json_url,
    output_dir="tiles",
    tile_width=520,
    tile_height=520,
):
    item = requests.get(json_url).json()
    assets = item["assets"]

    # Get the visual image
    visual_asset = assets.get("visual")

    if visual_asset is None:
        raise ValueError("No 'visual' asset found.")

    visual_url = urljoin(json_url, visual_asset["href"])

    # Find the cloud-mask asset
    cloud_asset = None

    for key, asset in assets.items():
        key_lower = key.lower()
        roles = [role.lower() for role in asset.get("roles", [])]

        if (
            "cloud" in key_lower
            or "mask" in key_lower
            or "cloud" in roles
        ):
            cloud_asset = asset
            break

    if cloud_asset is None:
        raise ValueError("No cloud-mask asset found.")

    cloud_mask_url = urljoin(json_url, cloud_asset["href"])

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save the complete cloud mask once
    #mask_path = output_dir / "cloud_mask.tif"

    #with requests.get(cloud_mask_url, stream=True) as response:
    #   response.raise_for_status()

     #   with open(mask_path, "wb") as file:
       #     for chunk in response.iter_content(chunk_size=1024 * 1024):
     #           file.write(chunk)

    saved_tiles = []

  # Display visual image tiles in a grid
    with rasterio.open(visual_url) as src:
        rows = math.ceil(src.height / tile_height)
        cols = math.ceil(src.width / tile_width)

        fig, axes = plt.subplots(
            rows,
            cols,
            figsize=(4 * cols, 4 * rows),
            squeeze=False,
        )

        for row in range(rows):
            for col in range(cols):
                x = col * tile_width
                y = row * tile_height

                width = min(tile_width, src.width - x)
                height = min(tile_height, src.height - y)

                window = rasterio.windows.Window(
                    x,
                    y,
                    width,
                    height,
                )

                data = src.read(window=window)

                ax = axes[row][col]

                if data.shape[0] >= 3:
                    # Rasterio format is bands, height, width.
                    # Matplotlib expects height, width, bands.
                    image = np.moveaxis(data[:3], 0, -1)

                    # Normalize for display if the image is uint16.
                    image = image.astype(float)
                    image -= image.min()

                    if image.max() > 0:
                        image /= image.max()

                    ax.imshow(image)

                else:
                    ax.imshow(data[0], cmap="gray")

                ax.set_title(
                    f"row={row}, col={col}"
                )
                ax.axis("off")

        # Hide unused axes, if any
        for index in range(rows * cols):
            row = index // cols
            col = index % cols

            if row >= rows or col >= cols:
                axes[row][col].axis("off")

        plt.tight_layout()
        plt.show()


In [ ]:
slice_visual_save_mask("https://maxar-opendata.s3.amazonaws.com/events/Cyclone-Chido-Dec15/ard/38/211311112122/2024-09-13/104001009990E600.json")

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

mask_path = "https://maxar-opendata.s3.amazonaws.com/events/Cyclone-Chido-Dec15/ard/38/211311112122/2024-09-13/104001009990E600-clouds.tif"

with rasterio.open(mask_path) as src:
    mask = src.read(1)

    print("Shape:", mask.shape)
    print("Unique values:", np.unique(mask))

plt.figure(figsize=(10, 10))
plt.imshow(mask, cmap="gray", interpolation="nearest")
plt.colorbar(label="Mask value")
plt.title("Cloud Mask")
plt.axis("off")
plt.show()

In [ ]:
slice_visual_save_mask("https://maxar-opendata.s3.amazonaws.com/events/cyclone-emnati22/ard/38/300222330301/2022-02-28/105001002A31C400.json")

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

mask_path = "https://maxar-opendata.s3.amazonaws.com/events/cyclone-emnati22/ard/38/300222330301/2022-02-28/105001002A31C400-clouds.tif"

with rasterio.open(mask_path) as src:
    mask = src.read(1)

    print("Shape:", mask.shape)
    print("Unique values:", np.unique(mask))

plt.figure(figsize=(10, 10))
plt.imshow(mask, cmap="gray", interpolation="nearest")
plt.colorbar(label="Mask value")
plt.title("Cloud Mask")
plt.axis("off")
plt.show()